# Portfolio Returns — Example

Fetch aligned daily returns for a weighted portfolio from yfinance, using the `portfolio_vol` module in this folder.

Downstream cells (vol, correlation) can be dropped in once the returns panel is built.

## 1. One-time setup

Uncomment and run the next cell once to install dependencies into the active Python env. After that you can re-run the notebook without it.

In [ ]:
# %pip install -r requirements.txt

## 2. Define the portfolio

Edit the dict — keys are tickers (case-insensitive), values are weights. They must sum to 1.0, or pass `normalize=True` to rescale automatically.

In [ ]:
from portfolio_vol import Portfolio, fetch_returns

WEIGHTS = {
    "SPY": 0.50,
    "QQQ": 0.30,
    "TLT": 0.20,
}

START = "2024-01-01"
END   = "2025-01-01"

portfolio = Portfolio.from_weights(WEIGHTS)
portfolio.weights

## 3. Fetch daily returns

Returns a DataFrame of daily log returns (switch to `method="simple"` if you prefer). Rows are aligned on the intersection of trading days across tickers.

In [ ]:
returns = fetch_returns(portfolio, start=START, end=END)
print(f"shape: {returns.shape}")
returns.head()

In [ ]:
returns.tail()

## 4. Quick sanity checks

These are placeholders — you'll replace them with your realized-vol and correlation code.

In [ ]:
# Annualized stdev per name (252 trading days), as a rough vol proxy
ann_vol = returns.std() * (252 ** 0.5)
ann_vol.rename("ann_stdev")

In [ ]:
# Sample correlation matrix
returns.corr()

In [ ]:
# Daily portfolio return series (returns dot weights)
port_ret = returns @ portfolio.weights
port_ret.describe()

## 5. Error handling — what to expect

- `PortfolioError` — bad spec (weights don't sum to 1, duplicate ticker, non-finite weight, empty dict).
- `FetchError` — yfinance issue (bad ticker, no overlap across names, all retries failed, too few observations).

Try swapping in a bogus ticker below to see the error path.

In [ ]:
from portfolio_vol import FetchError, PortfolioError

try:
    bad = Portfolio.from_weights({"SPY": 0.5, "NOTATICKER": 0.5})
    fetch_returns(bad, start=START, end=END)
except (FetchError, PortfolioError) as e:
    print(f"{type(e).__name__}: {e}")